!pip install --upgrade pip
!pip install whisperx
!brew install ffmpeg
!pip install ffmpeg-python
!pip install OpenCC

In [1]:
from huggingface_hub import snapshot_download

# 下載模型到指定資料夾
model_path = snapshot_download(
    repo_id="tiantiaf/voxlect-tibetan-dialect-whisper-large-v3",
    local_dir="voxlect-main/tibetan-whisper-model",
    local_dir_use_symlinks=False
)

print(f"模型已下載至: {model_path}")

/Users/pfhsu/miniforge3/envs/whisper-jax/lib/python3.12/site-packages/huggingface_hub/file_download.py:834: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

模型已下載至: /Users/pfhsu/PycharmProjects/whisper-jax/tibetan-whisper-model


In [1]:
# Load model directly
from transformers import AutoModel
model = AutoModel.from_pretrained("tiantiaf/voxlect-tibetan-dialect-whisper-large-v3", dtype="auto")

KeyboardInterrupt: 

In [2]:
import ffmpeg
import math
import os
from pathlib import Path
import json
import opencc
import datetime
import glob

In [3]:
# Add mp3 audio file name
mp3_audio = "910810b_P077.mp3"

In [4]:
trim_duration = 60 * 10
output_dir = "audio"
os.makedirs("audio", exist_ok=True)
probe = ffmpeg.probe(mp3_audio)
stream = next((stream for stream in probe['streams'] if stream['codec_type'] == 'audio'), None)
duration = math.floor(float(stream['duration']))
print(f"Total duration: {duration}")
print(f"Trim into {math.ceil(duration/trim_duration)} files by {trim_duration} mins")

audio_input = ffmpeg.input(mp3_audio)
flag = 0
for cut in range(math.ceil(duration/trim_duration)):
    if flag+trim_duration > duration:
        end = duration
    else:
        end = flag + trim_duration
    audio_cut = audio_input.audio.filter('atrim', start=flag, end=end)
    audio_output = ffmpeg.output(audio_cut, f'audio/{cut}_{flag}_{end}.mp3')
    ffmpeg.run(audio_output)
    flag += trim_duration


Total duration: 3774
Trim into 7 files by 600 mins


ffmpeg version 5.1.6 Copyright (c) 2000-2024 the FFmpeg developers
  built with Apple clang version 16.0.0 (clang-1600.0.26.6)
  configuration: --prefix='/usr/local/Cellar/ffmpeg@5/5.1.6_7' --datadir='/usr/local/Cellar/ffmpeg@5/5.1.6_7/share/ffmpeg' --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-

Error: ffmpeg error (see stderr output for detail)

In [5]:
import os
from pathlib import Path
import warnings
import logging

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("XDG_CACHE_HOME", str((Path.cwd() / ".cache").resolve()))
os.environ.setdefault("MPLCONFIGDIR", str((Path.cwd() / ".mplconfig").resolve()))
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

warnings.filterwarnings(
    "ignore",
    message=r"\s*torchcodec is not installed correctly so built-in audio decoding will fail\.",
    module=r"pyannote\.audio\.core\.io",
)
warnings.filterwarnings(
    "ignore",
    message=r"torchaudio\._backend\.list_audio_backends has been deprecated\..*",
    module=r"speechbrain\.utils\.torch_audio_backend",
)
logging.getLogger("speechbrain.utils.quirks").setLevel(logging.ERROR)

import whisperx


In [6]:
device = "cpu"
batch_size = 16 # reduce if low on GPU mem
compute_type = "int8" # change to "int8" if low on GPU mem (may reduce accuracy)

# 1. Transcribe with original whisper (batched)

model = whisperx.load_model("deepdml/faster-whisper-large-v3-turbo-ct2",
                            device, download_root='models', language="zh",
                            compute_type=compute_type,
                            )

#model = whisperx.load_model("openpecha_op-whisper_small-ft-v2",
#                            device, download_root='models', language="bo",
#                            compute_type=compute_type,
#                            )

# save model to local path (optional)
# model_dir = "/path/"
# model = whisperx.load_model("large-v2", device, compute_type=compute_type, download_root=model_dir)

2026-03-19 01:51:26 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.5.0.post0. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../miniforge3/envs/whisper-jax/lib/python3.12/site-packages/whisperx/assets/pytorch_model.bin`


In [7]:
audio_cuts = glob.glob("audio/*.mp3")
audio_cuts = sorted(audio_cuts, key=lambda x: x)
print(audio_cuts)
print(f"find totally audio_cut: {len(audio_cuts)}")

['audio/0_0_600.mp3', 'audio/1_600_1200.mp3', 'audio/2_1200_1800.mp3', 'audio/3_1800_2400.mp3', 'audio/4_2400_3000.mp3', 'audio/5_3000_3600.mp3', 'audio/6_3600_3774.mp3']
find totally audio_cut: 7


In [8]:
audio_cuts = audio_cuts[4:]

In [10]:
import pyewts
converter = pyewts.pyewts()
os.makedirs("json", exist_ok=True)
for index, audio_cut in enumerate(audio_cuts):
    print(index)
    print(f"Start transcribe: {audio_cut}")
    audio = whisperx.load_audio(audio_cut, sr=16000)
    result = model.transcribe(audio, batch_size=batch_size)
    print(result["segments"]) # before alignment
    with open(f"json/{Path(audio_cut).stem}.json", "w", encoding='utf8') as f:
        json.dump(result["segments"], f, indent=4, ensure_ascii=False)

# delete model if low on GPU resources
# import gc; gc.collect(); torch.cuda.empty_cache(); del model

0
Start transcribe: audio/4_2400_3000.mp3
[{'text': '心理的那特别是说不会死的人你看如果在座的又健康又年轻的话那这个好像说我根本轮不到我的那种心一定更强烈所以这边要修的是说不管你是年老还是年轻而是说你在你心理中着实有这种我一定会死的这种感受生起', 'start': 0.031, 'end': 19.64}, {'text': '问题这个感受就很困难了为什么你想一想这个他的排列第一决定死 第二个死无定期他这个是以感受 宗大师是以感受而编列的那这个多少人看得出来 看不出来因为宗大师一直在讲这个道次第数量次第一定要决定 决定以后要照着这样这个是一个心路历程你如果没有决定死', 'start': 19.64, 'end': 45.138}, {'text': '有生起的话,你死无定期,你不会生起感受的那何以见的,比如格西举两个例子比如说你有一个亲友,父母亲友医师在判断说,这些绝症了,没我啊', 'start': 45.138, 'end': 59.043}, {'text': '那我们的亲友,第二句话是什么?当席,一定是这样子嘛,何时呢?对不对?那是一个,第二个,比如说格西达赖喇嘛派到台湾,然后就忽然有摇散说不能呆了,要回印度了,那你格西也不知道,然后去追问,真的是了?那这个时候,格西第二个问题是什么?我要回印度了,当席', 'start': 59.448, 'end': 83.866}, {'text': '有没有 何时嘛这个心一定是这样走的嘛所以你看我们第一个当你心中对决定一定会死的时候第一个是什么当时嘛所以第二个就是何时会死嘛死了死的时候没有一定哦不是因为你年轻就不死哦所以第二个就是何时', 'start': 83.866, 'end': 101.804}, {'text': '所以这边要看到的是宗大师的编配全部都是我们心的心路历程那这个东西不是只是说词意上而是你用感受上的那种走法所以说为什么那个赤地数量那么重要而且那种感受那么重要就是在这里我想 我在三世读书 哈利波提起来有的坐在里面吗你也不知道去我三八十', 'start': 101.804, 'end': 128.669}, {'text': '每一天都在一起,每一天都在一起,每一天都在一起,每一天都在一起,每一天都在一起,每一天都在一起,每一天都在一起,每一

In [11]:
os.makedirs("text", exist_ok=True)
json_cuts = glob.glob("json/*.json")
json_cuts = sorted(json_cuts, key=lambda x: x)
print(json_cuts)
print(f"find totally json_cuts: {len(json_cuts)}")
for index, json_cut in enumerate(json_cuts):
    print(f"Start transcribe: {json_cut}")
    start = int(str.split(Path(json_cut).stem, "_")[1])
    end = int(str.split(Path(json_cut).stem, "_")[2])
    with open(json_cut, 'r', encoding='utf8') as file:
        result = json.load(file)
    with open(f"text/{Path(json_cut).stem}.txt", "w", encoding='utf8') as f:
        for s in result:
            sec_s = int(s["start"])
            sec_e = int(s["end"])
            time_line = f"{datetime.timedelta(seconds=start) + datetime.timedelta(seconds=sec_s)}-{datetime.timedelta(seconds=start) + datetime.timedelta(seconds=sec_e)}"
            print(time_line)
            f.write(time_line+ '\n')
            print(s['text'])
            f.writelines(s['text'] + '\n')

['json/4_2400_3000.json', 'json/5_3000_3600.json', 'json/6_3600_3774.json']
find totally json_cuts: 3
Start transcribe: json/4_2400_3000.json
0:40:00-0:40:19
心理的那特别是说不会死的人你看如果在座的又健康又年轻的话那这个好像说我根本轮不到我的那种心一定更强烈所以这边要修的是说不管你是年老还是年轻而是说你在你心理中着实有这种我一定会死的这种感受生起
0:40:19-0:40:45
问题这个感受就很困难了为什么你想一想这个他的排列第一决定死 第二个死无定期他这个是以感受 宗大师是以感受而编列的那这个多少人看得出来 看不出来因为宗大师一直在讲这个道次第数量次第一定要决定 决定以后要照着这样这个是一个心路历程你如果没有决定死
0:40:45-0:40:59
有生起的话,你死无定期,你不会生起感受的那何以见的,比如格西举两个例子比如说你有一个亲友,父母亲友医师在判断说,这些绝症了,没我啊
0:40:59-0:41:23
那我们的亲友,第二句话是什么?当席,一定是这样子嘛,何时呢?对不对?那是一个,第二个,比如说格西达赖喇嘛派到台湾,然后就忽然有摇散说不能呆了,要回印度了,那你格西也不知道,然后去追问,真的是了?那这个时候,格西第二个问题是什么?我要回印度了,当席
0:41:23-0:41:41
有没有 何时嘛这个心一定是这样走的嘛所以你看我们第一个当你心中对决定一定会死的时候第一个是什么当时嘛所以第二个就是何时会死嘛死了死的时候没有一定哦不是因为你年轻就不死哦所以第二个就是何时
0:41:41-0:42:08
所以这边要看到的是宗大师的编配全部都是我们心的心路历程那这个东西不是只是说词意上而是你用感受上的那种走法所以说为什么那个赤地数量那么重要而且那种感受那么重要就是在这里我想 我在三世读书 哈利波提起来有的坐在里面吗你也不知道去我三八十
0:42:09-0:42:30
每一天都在一起,每一天都在一起,每一天都在一起,每一天都在一起,每一天都在一起,每一天都在一起,每一天都在一起,每一天都在一起,每一天都在一起
0:42:31-0:42:57
那些人在家里面有些人在家里面有些人在家里面有些人在家里面有些人在家里面有些人在

In [12]:
converter = opencc.OpenCC('s2tw.json')
import re

def sort_key(path):
    m = re.search(r'text/(\d+)_', path)
    return int(m.group(1)) if m else float('inf')
txt_cuts = sorted(glob.glob("text/*.txt"), key=sort_key)
print(txt_cuts)
print(f"find totally txt_cuts: {len(txt_cuts)}")
print(txt_cuts)
for index, txt_cut in enumerate(txt_cuts):
    print(f"Start merged transcribe: {txt_cut}")
    with open(txt_cut, 'r', encoding='utf8') as file:
        txt = file.read()
    with open(f"{mp3_audio}.txt", "a+", encoding='utf8') as f:
        f.write(converter.convert(txt))

['text/4_2400_3000.txt', 'text/5_3000_3600.txt', 'text/6_3600_3774.txt']
find totally txt_cuts: 3
['text/4_2400_3000.txt', 'text/5_3000_3600.txt', 'text/6_3600_3774.txt']
Start merged transcribe: text/4_2400_3000.txt
Start merged transcribe: text/5_3000_3600.txt
Start merged transcribe: text/6_3600_3774.txt


In [2]:
import glob
audio_cuts = glob.glob("audio/*.mp3")
audio_cuts = sorted(audio_cuts, key=lambda x: x)
print(audio_cuts)
print(f"find totally audio_cut: {len(audio_cuts)}")

['audio/0_0_600.mp3', 'audio/1_600_1200.mp3', 'audio/2_1200_1800.mp3', 'audio/3_1800_2400.mp3', 'audio/4_2400_3000.mp3', 'audio/5_3000_3600.mp3', 'audio/6_3600_3780.mp3']
find totally audio_cut: 7


In [3]:
import os
from pathlib import Path
import warnings
import logging

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("XDG_CACHE_HOME", str((Path.cwd() / ".cache").resolve()))
os.environ.setdefault("MPLCONFIGDIR", str((Path.cwd() / ".mplconfig").resolve()))
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

warnings.filterwarnings(
    "ignore",
    message=r"\s*torchcodec is not installed correctly so built-in audio decoding will fail\.",
    module=r"pyannote\.audio\.core\.io",
)
warnings.filterwarnings(
    "ignore",
    message=r"torchaudio\._backend\.list_audio_backends has been deprecated\..*",
    module=r"speechbrain\.utils\.torch_audio_backend",
)
logging.getLogger("speechbrain.utils.quirks").setLevel(logging.ERROR)

import torch
import whisperx

device = "cpu"
batch_size = 16
compute_type = "int8"
models = {
    "zh": whisperx.load_model("deepdml/faster-whisper-large-v3-turbo-ct2", device, language="zh", compute_type=compute_type),
    "bo": whisperx.load_model("openpecha_op-whisper_small-ft-v2", device, language="bo", compute_type=compute_type)
}

align_config = {
    "zh": whisperx.load_align_model(language_code="zh", device=device, model_name="jonatasgrosman/wav2vec2-large-xlsr-53-chinese-zh-cn"),
    "bo": whisperx.load_align_model(language_code="bo", device=device, model_name="openpecha/op-wav2vec2-ft-v17")
}

audio = whisperx.load_audio(audio_cuts[0])
vad_model = whisperx.vads.pyannote.load_vad_model(device)



/Users/pfhsu/miniforge3/envs/whisper-jax/lib/python3.12/site-packages/speechbrain/utils/torch_audio_backend.py:54: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
INFO:speechbrain.utils.quirks:Applied quirks (see `speechbrain.utils.quirks`): [disable_jit_profiling, allow_tf32]
INFO:speechbrain.utils.quirks:Excluded quirks specified by the `SB_DISABLE_QUIRKS` environment (comma-separated list): []


2026-03-09 21:03:47 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-03-09 21:03:47 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.5.0.post0. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../miniforge3/envs/whisper-jax/lib/python3.12/site-packages/whisperx/assets/pytorch_model.bin`


2026-03-09 21:03:48 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-03-09 21:03:48 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.5.0.post0. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../miniforge3/envs/whisper-jax/lib/python3.12/site-packages/whisperx/assets/pytorch_model.bin`
/Users/pfhsu/miniforge3/envs/whisper-jax/lib/python3.12/site-packages/transformers/configuration_utils.py:315: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.5.0.post0. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../miniforge3/envs/whisper-jax/lib/python3.12/site-packages/whisperx/assets/pytorch_model.bin`


In [4]:
vad_segments = vad_model({"waveform": torch.from_numpy(audio).unsqueeze(0), "sample_rate": 16000})

In [7]:
from pyannote.audio.utils.signal import Binarize
# 也可用: from whisperx.vads.pyannote import Binarize

# 會回傳 Annotation（真正的語音區段）
speech_ann = Binarize(
    onset=0.5,
    offset=0.4,
    min_duration_on=0.5,   # 直接過濾短片段
)(vad_segments)


final_segments = list(speech_ann.get_timeline())

for segment in final_segments:
    start_sec = segment.start
    end_sec = segment.end
    dur = end_sec - start_sec
    if dur < 0.5:
        continue

    start_samp = max(0, int(start_sec * SAMPLE_RATE))
    end_samp = min(len(audio), int(end_sec * SAMPLE_RATE))
    audio_chunk = audio[start_samp:end_samp]
    if len(audio_chunk) == 0:
        continue

    # WhisperX pipeline API: detect_language(audio_np) -> str
    detected_lang = models["bo"].detect_language(audio_chunk)
    if detected_lang.startswith("zh"):
        detected_lang = "zh"

    model_to_use = models.get(detected_lang, models["zh"])
    model_a, metadata = align_config.get(detected_lang, align_config["zh"])

    result = model_to_use.transcribe(audio_chunk, batch_size=batch_size)

    if not result.get("segments"):
        print(f"[{start_sec:.2f}s - {end_sec:.2f}s] 偵測語言: {detected_lang}, 無可用文字")
        continue

    aligned = whisperx.align(
        result["segments"], model_a, metadata, audio_chunk, device, return_char_alignments=False
    )

    # 將 chunk 內時間軸轉回原始音檔時間軸
    for s in aligned["segments"]:
        s["start"] = round(s["start"] + start_sec, 3)
        s["end"] = round(s["end"] + start_sec, 3)

    text_preview = aligned["segments"][0]["text"] if aligned["segments"] else ""
    print(f"[{start_sec:.2f}s - {end_sec:.2f}s] 偵測語言: {detected_lang}, 內容: {text_preview}")
    print(aligned["segments"])


2026-03-09 21:09:49 - whisperx.asr - WARNING - Audio is shorter than 30s, language detection may be inaccurate
2026-03-09 21:09:50 - whisperx.asr - INFO - Detected language: bo (0.61) in first 30s of audio
2026-03-09 21:09:50 - whisperx.asr - WARNING - Audio is shorter than 30s, language detection may be inaccurate
2026-03-09 21:09:51 - whisperx.asr - INFO - Detected language: bo (0.61) in first 30s of audio
[0.03s - 0.99s] 偵測語言: bo, 內容: a le sngon la ya/__
[{'start': 0.062, 'end': 1.045, 'text': 'a le sngon la ya/__', 'words': [{'word': 'a', 'start': np.float64(0.031), 'end': np.float64(0.052), 'score': np.float64(0.015)}, {'word': 'le', 'start': np.float64(0.074), 'end': np.float64(0.117), 'score': np.float64(0.696)}, {'word': 'sngon', 'start': np.float64(0.138), 'end': np.float64(0.245), 'score': np.float64(0.775)}, {'word': 'la', 'start': np.float64(0.266), 'end': np.float64(0.33), 'score': np.float64(0.562)}, {'word': 'ya/__', 'start': np.float64(0.352), 'end': np.float64(1.014), 

KeyboardInterrupt: 

# 3. Assign speaker labels
diarize_model = whisperx.DiarizationPipeline(use_auth_token="YOUR_HF_TOKEN", device=device)

# add min/max number of speakers if known
diarize_segments = diarize_model(audio)
diarize_model(audio, min_speakers=2, max_speakers=2)

result = whisperx.assign_word_speakers(diarize_segments, result)
print(diarize_segments)
print(result["segments"]) # segments are now assigned speaker IDs